In [0]:
### complex datatypes
### json 
### user defined functions
### spark sql
### file formats
### paritions
### structure streaming 
### unity catelog 
### delta lake


In [0]:
# complex data types
from pyspark.sql import SparkSession 
from pyspark.sql.functions import * 
from pyspark.sql.types import *
spark=SparkSession.builder.appName("complex datatypes examples").getOrCreate()

In [0]:
# Array type:
# store multiple values of the same data type in single column

In [0]:
# example 1
schema=StructType([
    StructField("id",IntegerType()),
    StructField("skills",ArrayType(StringType()))
])

data=[
    (1,["python","sql"]),
    (2,["pyspark","Azure"])
]
df=spark.createDataFrame(data,schema)
df.show(truncate=False)
df.printSchema()

+---+----------------+
|id |skills          |
+---+----------------+
|1  |[python, sql]   |
|2  |[pyspark, Azure]|
+---+----------------+

root
 |-- id: integer (nullable = true)
 |-- skills: array (nullable = true)
 |    |-- element: string (containsNull = true)



In [0]:
# example 2
schema=StructType([
    StructField("student",StringType()),
    StructField("marks",ArrayType(IntegerType()))
])
data=[
    ("raju",[80,85,90]),
    ("buchi",[85,90,95])
]

df= spark.createDataFrame(data,schema)
df.show(truncate=False)


+-------+------------+
|student|marks       |
+-------+------------+
|raju   |[80, 85, 90]|
|buchi  |[85, 90, 95]|
+-------+------------+



In [0]:
# example 3 : array of dates as strings
schema=StructType([
    StructField("emp_id",IntegerType()),
    StructField("login_dates",ArrayType(StringType()))
])
data=[
    (101,["2026-06-01","2026-06-02"]),
    (102,["2026-06-03","2026-06-04"])
]
df=spark.createDataFrame(data,schema)
df.show(truncate=False)

+------+------------------------+
|emp_id|login_dates             |
+------+------------------------+
|101   |[2026-06-01, 2026-06-02]|
|102   |[2026-06-03, 2026-06-04]|
+------+------------------------+



In [0]:
# example 4
schema=StructType([
    StructField("id",IntegerType()),
    StructField("Scores",ArrayType(IntegerType(),True))
])
data=[
    (1,[10,20,None]),
    (2,[30,None,50])
]
df=spark.createDataFrame(data,schema)
df.show(truncate=False)

+---+--------------+
|id |Scores        |
+---+--------------+
|1  |[10, 20, NULL]|
|2  |[30, NULL, 50]|
+---+--------------+



In [0]:
# Example 5:Nested structtype
schema=StructType([
    StructField("id",IntegerType()),
    StructField("address",StructType([
        StructField("city",StringType()),
        StructField("state",StringType())
    ]))
])
data=[
    (1,("Kurnool","Andhra Pradesh")),
    (2,("Bangalore","Karnataka"))
]

df=spark.createDataFrame(data,schema)
df.show(truncate=False)

+---+-------------------------+
|id |address                  |
+---+-------------------------+
|1  |{Kurnool, Andhra Pradesh}|
|2  |{Bangalore, Karnataka}   |
+---+-------------------------+



In [0]:
df.select("id","address.city","address.state").show()

+---+---------+--------------+
| id|     city|         state|
+---+---------+--------------+
|  1|  Kurnool|Andhra Pradesh|
|  2|Bangalore|     Karnataka|
+---+---------+--------------+



In [0]:
# example 6:struct inside array type
schema=StructType([
    StructField("order_id",IntegerType()),
    StructField("items",ArrayType(
        StructType([
            StructField("product",StringType()),
            StructField("quantity",IntegerType())
        ])
    ))
])

data=[
    (1,[("Laptop",1),("Mouse",2)]),
    (2,[("Mobile",3),("Charger",4)])
]
df=spark.createDataFrame(data,schema)
df.show(truncate=False)

+--------+---------------------------+
|order_id|items                      |
+--------+---------------------------+
|1       |[{Laptop, 1}, {Mouse, 2}]  |
|2       |[{Mobile, 3}, {Charger, 4}]|
+--------+---------------------------+



In [0]:
df.select(col("order_id"),col("items")[0]["quantity"].alias("order quantity")).show()

+--------+--------------+
|order_id|order quantity|
+--------+--------------+
|       1|             1|
|       2|             3|
+--------+--------------+



In [0]:
df.select(col("order_id"),col("items")[1]["quantity"].alias("order quantity")).show()

+--------+--------------+
|order_id|order quantity|
+--------+--------------+
|       1|             2|
|       2|             4|
+--------+--------------+



In [0]:
# example 7:map type
# key value pairs (dictionary):json object,each key in map is unique


In [0]:
from pyspark.sql import SparkSession 
from pyspark.sql.types import *
spark=SparkSession.builder.getOrCreate()

In [0]:
data=[
    (101,{"python":5,"SQL":4,"Azure":7}),
    (102,{"Java":9,"Spark":2}),
    (103,{"AWS":5,"Docker":1,"Kubernetes":3})
]

schema=StructType([
    StructField("emp_id",IntegerType()),
    StructField("skills",MapType(StringType(),IntegerType()))
])

df=spark.createDataFrame(data,schema)
df.show(truncate=False)
df.printSchema()

+------+----------------------------------------+
|emp_id|skills                                  |
+------+----------------------------------------+
|101   |{python -> 5, SQL -> 4, Azure -> 7}     |
|102   |{Java -> 9, Spark -> 2}                 |
|103   |{AWS -> 5, Docker -> 1, Kubernetes -> 3}|
+------+----------------------------------------+

root
 |-- emp_id: integer (nullable = true)
 |-- skills: map (nullable = true)
 |    |-- key: string
 |    |-- value: integer (valueContainsNull = true)



In [0]:
df.select(col("emp_id"),col("skills")["Spark"]).show()

df.select(col("emp_id"),col("skills")["spark"]).show()  # its case sensetive

+------+-------------+
|emp_id|skills[Spark]|
+------+-------------+
|   101|         NULL|
|   102|            2|
|   103|         NULL|
+------+-------------+

+------+-------------+
|emp_id|skills[spark]|
+------+-------------+
|   101|         NULL|
|   102|         NULL|
|   103|         NULL|
+------+-------------+



In [0]:
from pyspark.sql.functions import map_keys,map_values
df.select("emp_id",map_keys("skills").alias("skills"),map_values("skills").alias("skills ratings")).show()

+------+--------------------+--------------+
|emp_id|              skills|skills ratings|
+------+--------------------+--------------+
|   101|[python, SQL, Azure]|     [5, 4, 7]|
|   102|       [Java, Spark]|        [9, 2]|
|   103|[AWS, Docker, Kub...|     [5, 1, 3]|
+------+--------------------+--------------+



In [0]:
# using explode it separates the keys and values
from pyspark.sql.functions import explode

In [0]:
df.select("emp_id",explode("skills")).show()

+------+----------+-----+
|emp_id|       key|value|
+------+----------+-----+
|   101|    python|    5|
|   101|       SQL|    4|
|   101|     Azure|    7|
|   102|      Java|    9|
|   102|     Spark|    2|
|   103|       AWS|    5|
|   103|    Docker|    1|
|   103|Kubernetes|    3|
+------+----------+-----+



In [0]:
from pyspark.sql import SparkSession 
from pyspark.sql.functions import explode 

In [0]:
spark=SparkSession.builder.getOrCreate()
data=[
    (1,["python","SQL","Azure"]),
    (2,["Java","Spark"]),
    (3,["AWS"])
]


df=spark.createDataFrame(data,["id","skills"])
df.show(truncate=False)

+---+--------------------+
|id |skills              |
+---+--------------------+
|1  |[python, SQL, Azure]|
|2  |[Java, Spark]       |
|3  |[AWS]               |
+---+--------------------+



In [0]:
df.select("ID",explode("skills").alias("professional_skills")).show()

+---+-------------------+
| ID|professional_skills|
+---+-------------------+
|  1|             python|
|  1|                SQL|
|  1|              Azure|
|  2|               Java|
|  2|              Spark|
|  3|                AWS|
+---+-------------------+



In [0]:
data=[
    (101,[10,20,30]),
    (102,[40,50])
]
df=spark.createDataFrame(data,["id","marks"])
df.show(truncate=False)

+---+------------+
|id |marks       |
+---+------------+
|101|[10, 20, 30]|
|102|[40, 50]    |
+---+------------+



In [0]:
df.select("id",explode("marks").alias("marks")).show()

+---+-----+
| id|marks|
+---+-----+
|101|   10|
|101|   20|
|101|   30|
|102|   40|
|102|   50|
+---+-----+



In [0]:
from pyspark.sql.types import *
data=[
    (1,{"python":2,"SQL":4}),
    (2,{"Java":6,"Spark":8})
]

schema=StructType([
    StructField("id",IntegerType()),
    StructField("Skills",MapType(StringType(),IntegerType()))

])
df=spark.createDataFrame(data,schema)
df.show(truncate=False)
df.select("id",explode("skills")).show()

+---+-----------------------+
|id |Skills                 |
+---+-----------------------+
|1  |{python -> 2, SQL -> 4}|
|2  |{Java -> 6, Spark -> 8}|
+---+-----------------------+

+---+------+-----+
| id|   key|value|
+---+------+-----+
|  1|python|    2|
|  1|   SQL|    4|
|  2|  Java|    6|
|  2| Spark|    8|
+---+------+-----+



In [0]:
from pyspark.sql import SparkSession 
from pyspark.sql.functions import explode

spark=SparkSession.builder.getOrCreate()

data=[
    (1,[["python","sql"],["spark","azure"]]),
    (2,[["java","kafka"]])
]
df=spark.createDataFrame(data,["id","skills"])
df.show(truncate=False)

+---+-------------------------------+
|id |skills                         |
+---+-------------------------------+
|1  |[[python, sql], [spark, azure]]|
|2  |[[java, kafka]]                |
+---+-------------------------------+



In [0]:
df.select(col("id"),col("skills")[0]).show() # extract indexing 0
df.select(col("id"),col("skills")[0][0]).show()# nested indexing 0,0
df.select(col("id"),col("skills")[0][1]).show()# nested indexing 0,1


+---+-------------+
| id|    skills[0]|
+---+-------------+
|  1|[python, sql]|
|  2|[java, kafka]|
+---+-------------+

+---+------------+
| id|skills[0][0]|
+---+------------+
|  1|      python|
|  2|        java|
+---+------------+

+---+------------+
| id|skills[0][1]|
+---+------------+
|  1|         sql|
|  2|       kafka|
+---+------------+



In [0]:
df.select(col("id"),get(col("skills"),1)).show()

+---+--------------+
| id|get(skills, 1)|
+---+--------------+
|  1|[spark, azure]|
|  2|          NULL|
+---+--------------+



In [0]:
df1=df.select("id",explode("skills").alias("skill group"))
df1.show()
df1.select("id",explode("skill group").alias("skills")).show()



+---+--------------+
| id|   skill group|
+---+--------------+
|  1| [python, sql]|
|  1|[spark, azure]|
|  2| [java, kafka]|
+---+--------------+

+---+------+
| id|skills|
+---+------+
|  1|python|
|  1|   sql|
|  1| spark|
|  1| azure|
|  2|  java|
|  2| kafka|
+---+------+



In [0]:
# example 8: poseexplode
from pyspark.sql.functions import posexplode
df1=df.select("id",explode("skills").alias("skill group"))
df1.show()
df1.select("id",posexplode("skill group")).alias("skills").show()

+---+--------------+
| id|   skill group|
+---+--------------+
|  1| [python, sql]|
|  1|[spark, azure]|
|  2| [java, kafka]|
+---+--------------+

+---+---+------+
| id|pos|   col|
+---+---+------+
|  1|  0|python|
|  1|  1|   sql|
|  1|  0| spark|
|  1|  1| azure|
|  2|  0|  java|
|  2|  1| kafka|
+---+---+------+



In [0]:
from pyspark.sql import SparkSession 
from pyspark.sql.functions import posexplode
spark = SparkSession.builder.getOrCreate()

data=[
    (101,["python","sql","spark"]),
    (102,["java","scala"])
]
df=spark.createDataFrame(data,["emp_id","skills"])
df.show()

df.select("emp_id",posexplode("skills").alias("positions","skills")).show()

+------+--------------------+
|emp_id|              skills|
+------+--------------------+
|   101|[python, sql, spark]|
|   102|       [java, scala]|
+------+--------------------+

+------+---------+------+
|emp_id|positions|skills|
+------+---------+------+
|   101|        0|python|
|   101|        1|   sql|
|   101|        2| spark|
|   102|        0|  java|
|   102|        1| scala|
+------+---------+------+



In [0]:
# example 9: Array zip
from pyspark.sql.functions import arrays_zip

data=[(
    ["python","sql","spark"],
    [5,4,3]
)]
df=spark.createDataFrame(data,["skills","ratings"])
df.show()

+--------------------+---------+
|              skills|  ratings|
+--------------------+---------+
|[python, sql, spark]|[5, 4, 3]|
+--------------------+---------+



In [0]:
df.select(arrays_zip("skills","ratings").alias("result")).show(truncate=False)

+-----------------------------------+
|result                             |
+-----------------------------------+
|[{python, 5}, {sql, 4}, {spark, 3}]|
+-----------------------------------+



In [0]:
x=[1,2,3]
y=[4,5,6]
list(zip(x,y))

[(1, 4), (2, 5), (3, 6)]

In [0]:
data=[(
    ["python","sql","java","spark"],
    [9,8,7]
)]
df=spark.createDataFrame(data,["skills","ratings"])
df.select(arrays_zip("skills","ratings").alias("result")).show(truncate=False)

+-------------------------------------------------+
|result                                           |
+-------------------------------------------------+
|[{python, 9}, {sql, 8}, {java, 7}, {spark, NULL}]|
+-------------------------------------------------+



In [0]:
import numpy as np 
data=np.random.randint(0, 10,(10, 10))
print(data)        # it gives 10*10 matrix

[[7 8 4 4 1 6 5 9 0 9]
 [1 4 4 6 4 5 2 2 9 1]
 [2 7 7 6 3 0 8 0 0 1]
 [2 2 6 0 2 8 8 1 8 8]
 [0 4 7 8 8 3 8 0 5 4]
 [2 6 0 1 3 7 1 3 3 9]
 [0 2 9 3 3 1 2 5 4 1]
 [2 1 3 0 9 1 2 8 3 1]
 [7 9 4 4 3 5 7 5 7 1]
 [0 6 6 8 5 0 2 2 9 9]]


In [0]:
data.flatten() # it gives flatten datat

array([7, 8, 4, 4, 1, 6, 5, 9, 0, 9, 1, 4, 4, 6, 4, 5, 2, 2, 9, 1, 2, 7,
       7, 6, 3, 0, 8, 0, 0, 1, 2, 2, 6, 0, 2, 8, 8, 1, 8, 8, 0, 4, 7, 8,
       8, 3, 8, 0, 5, 4, 2, 6, 0, 1, 3, 7, 1, 3, 3, 9, 0, 2, 9, 3, 3, 1,
       2, 5, 4, 1, 2, 1, 3, 0, 9, 1, 2, 8, 3, 1, 7, 9, 4, 4, 3, 5, 7, 5,
       7, 1, 0, 6, 6, 8, 5, 0, 2, 2, 9, 9])

In [0]:
from pyspark.sql.functions import flatten
data=[(
    [
    [1,2,3],
    [4,5,6],
    [7,8,9]
    ],
)]
df=spark.createDataFrame(data,["numbers"])
df.select(flatten("numbers").alias("result")).show(truncate=False)

+---------------------------+
|result                     |
+---------------------------+
|[1, 2, 3, 4, 5, 6, 7, 8, 9]|
+---------------------------+



In [0]:
data=[(
    [
        ["python","sql"],
        ["spark","ADF"],
        ["scala","kafka"]
    ],
)]
df=spark.createDataFrame(data,["skills"])
df.select(flatten("skills")).show(truncate=False)

+---------------------------------------+
|flatten(skills)                        |
+---------------------------------------+
|[python, sql, spark, ADF, scala, kafka]|
+---------------------------------------+



In [0]:
# example 10:array_contains 
from pyspark.sql.functions import array_contains 
data=[
    (101,["python","sql","spark"]),
    (102,["java","scala"])
]
df=spark.createDataFrame(data,["emp_id","skills"])
df.select("emp_id",array_contains("skills","python").alias("known_python")).show()

+------+------------+
|emp_id|known_python|
+------+------------+
|   101|        true|
|   102|       false|
+------+------------+



In [0]:
data=[
    ("order1",[101,102,103]),
    ("order2",[104,105])
]
df=spark.createDataFrame(data,["order_id","products"])

df.select("order_id",array_contains("products",105).alias("105 contains")).show()

+--------+------------+
|order_id|105 contains|
+--------+------------+
|  order1|       false|
|  order2|        true|
+--------+------------+

